# ST7 Project 2026

## Algorithm
1. Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import h5py
import os
import subprocess
import sys
import importlib
# from pysem import parse_sem3d_traces
from pathlib import Path

# sys.path.append(str(Path("pysem/src").resolve()))

path_to_src = str(Path("pysem/src").resolve())
print(f"Adding {path_to_src} to sys.path")
if path_to_src not in sys.path:
    sys.path.append(path_to_src)

from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from pysem.generate_h5_materials import write_h5
from pysem.parse_sem3d_snapshots import compute_gradients_main
from util_funct.sbatch_and_wait import sbatch_and_wait
from util_funct.compute_misfit import compute_misfit
from util_funct.read_stations_pos import read_stations_pos
from util_funct.write_backward_spec import write_backward_spec_from_template
from util_funct.write_misfit_files import write_time_reversed_residual_files

Adding /usr/users/cea_seism/tran_ngo/CEA_PROJECT/pysem/src to sys.path


## 1. Paths and parameters

In [ ]:
SEM3D_CONFIG_RES_FOLDER_PATH = "./sem3d_config_files"

FORWARD_PROBLEM_MESHER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "MESHER.sbatch")
FORWARD_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "SOLVER.sbatch")

TRACES_SIMULATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "traces")
TRACES_OSSERVATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "Uobs")

SEM3D_CONFIG_RES_FOLDER_PATH_ADJ = "./sem3d_config_files_adj"

ADJOINT_SOURCES_FOLDER_NAME = ""    #MUST BE short, otherwise sem3d will complain
ADJOINT_SOURCES_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, ADJOINT_SOURCES_FOLDER_NAME)

STATIONS_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "stations.txt")

BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "template/input_backward_template.spec")
ADJOINT_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "SOLVER_ADJOINT.sbatch")

## 2. Load observed data d_obs

In [ ]:
obs_stream = ParseSEM3DH5Traces(
    wkdir=TRACES_OSSERVATED_FOLDER_PATH,
    format='h5',
    names=['Uobs'],
    variables=['Displ'],
    components=['x', 'y', 'z']
)

obs_monitor = obs_stream['Uobs']

## 3. Initial material m_0

In [ ]:
! python3 ./pysem/src/pysem/generate_h5_materials.py @@prop "la" "mu" "ds" @@tag "linear_gradient" @@dir "z" @@xlim -2000 2000 @@ylim -2000 2000 @@zlim -2000 250 @@step 200 200 200 @@pfx 'example'
! mv example* ./sem3d_config_files/

Model tag: linear_gradient - Model grad: z
Domain limits/discretization : 
 X: -2000.0 m : 2000.0 m; nx=200 
 Y: -2000.0 m : 2000.0 m; ny=200 
 Z: -2000.0 m : 250.0 m; nz=200
Material files generated successfully!


## 4. Algorithm

In [ ]:
# sys.argv = ['parse_sem3d_snapshots.py', '@@wkd', './sem3d_config_files/res', '@@begin_time', '0', '@@end_time', '2']
from pysem.parse_sem3d_snapshots import compute_gradients_main

In [ ]:
N_ITER = 10 #to be modified

sbatch_and_wait(FORWARD_PROBLEM_MESHER_SBATCH_PATH)

stations = np.loadtxt(STATIONS_FILE_PATH) #read_stations_pos(STATIONS_FILE_PATH)

# for n in range(N_ITER):

# ── STEP 1 ────────────────────────────────────────
sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)

# ── STEP 2: MISFIT ────────────────────────────────────────────────────

J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, obs_monitor)

# ── STEP 3 ────────────────────────────────────────

time_reversed_residual = residual[::-1, :, :]

file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                t_sim, 
                                                OUTPUT_DIR=ADJOINT_SOURCES_FOLDER_PATH)

write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH, 
                                    output_backward_spec_path = SEM3D_CONFIG_RES_FOLDER_PATH_ADJ,
                                    adjoint_sources_folder_path = ADJOINT_SOURCES_FOLDER_PATH, 
                                    stations = stations, 
                                    file_names = file_names)

sbatch_and_wait(ADJOINT_PROBLEM_SOLVER_SBATCH_PATH)

print("")
# input.spec
# material

# ── STEP 4: GRADIENT ─────────────────────────────────────────────────
sys.argv = ['parse_sem3d_snapshots.py', '@@wkd', './sem3d_config_files/res', '@@begin_time', '0', '@@end_time', '2']
g_la, g_mu = compute_gradients_main()


    # ── STEP 5: CONJUGATE GRADIENT DIRECTION (Fletcher-Reeves) ────────────
    # if n == 0:
    #     p_la = -g_la
    #     p_mu = -g_mu
    # else:
    #     beta = (||g||^2) / (||g_prev||^2)
    #     p_la = -g_la + beta * p_la_prev
    #     p_mu = -g_mu + beta * p_mu_prev

    # ── STEP 6: LINE SEARCH (backtracking Armijo) ─────────────────────────
    # alpha = ALPHA_0
    # while True:
    #     m_la_trial = m_la + alpha * p_la
    #     m_mu_trial = m_mu + alpha * p_mu
    #     Write m_trial to HDF5
    #     Launch forward solve with m_trial  ← (BLACK BOX)
    #     Compute J_trial
    #     if J_trial < J + ARMIJO_TAU * alpha * (g · p): break
    #     alpha = ARMIJO_C * alpha

    # ── STEP 7: UPDATE ─────────────────────────────────────────────────────
    # m_la = m_la + alpha * p_la
    # m_mu = m_mu + alpha * p_mu
    # Write new material HDF5 files
    # Save current state (for restart)
    # p_la_prev, p_mu_prev = p_la, p_mu
    # g_prev = g

Submitted SOLVER.sbatch from /usr/users/cea_seism/tran_ngo/CEA_PROJECT/sem3d_config_files -> job 174619
Job 174619 finished with state: FAILED
SOLVER.sbatch executed in 18.07 seconds



RuntimeError: Job 174619 did not complete successfully: FAILED